In [1]:
from dotenv import load_dotenv, find_dotenv
from requests_oauthlib import OAuth2Session
from oauthlib.oauth2 import BackendApplicationClient
import truststore
import requests
import os
import json
import threading
import webview
import time

truststore.inject_into_ssl()
# Load the environment variables from the .env file
load_dotenv(find_dotenv())
from pandas import json_normalize



In [2]:
class OAuthManager:
    """
    Complete OAuth manager that handles initial auth AND refresh
    """
    def __init__(self, client_id, client_secret, redirect_uri, auth_url, token_url, scope):
        self.client_id = client_id
        self.client_secret = client_secret
        self.redirect_uri = redirect_uri
        self.auth_url = auth_url
        self.token_url = token_url
        self.scope = scope
        
        # These will be set after authentication
        self.oauth = None
        self.token = None
        self.access_token = None
        self.refresh_token = None
        self.token_expiry = None
    
    def authenticate(self):
        """
        Perform initial OAuth flow with browser popup
        Returns True if successful
        """
        # Create OAuth2 session
        self.oauth = OAuth2Session(
            self.client_id,
            redirect_uri=self.redirect_uri,
            scope=self.scope
        )
        
        # Generate authorization URL
        authorization_url, state = self.oauth.authorization_url(self.auth_url)
        
        print(f'Authenticating...')
        
        # Launch browser for user consent
        authorization_response = self._launch_browser_auth(authorization_url)
        
        if not authorization_response:
            print("✗ Authentication failed")
            return False
        
        # Exchange code for token
        self.token = self.oauth.fetch_token(
            self.token_url,
            authorization_response=authorization_response,
            client_secret=self.client_secret
        )
        
        # Store token details
        self.access_token = self.token["access_token"]
        self.refresh_token = self.token.get("refresh_token")
        self.token_expiry = time.time() + self.token.get("expires_in", 3600)
        
        print("✓ Authentication successful")
        return True
    
    def _launch_browser_auth(self, authorization_url):
        """Launch webview for OAuth authorization"""
        authorization_response = None
        
        def on_loaded():
            nonlocal authorization_response
            current_url = window.get_current_url()
            if current_url and self.redirect_uri in current_url:
                authorization_response = current_url
                print(f'✓ Captured Authorization')
                window.hide()
                time.sleep(2.5)
                window.destroy()
        
        window = webview.create_window(
            'OAuth Authorization',
            authorization_url,
            width=800,
            height=600,
            resizable=True,
            on_top=True
        )
        
        window.events.loaded += on_loaded
        webview.start()
        
        return authorization_response
    
    def _is_token_expired(self):
        """Check if access token is expired"""
        if not self.token_expiry:
            return True
        return time.time() >= (self.token_expiry - 60)
    
    def _refresh_access_token(self):
        """Refresh the access token"""
        if not self.refresh_token:
            print("⚠ No refresh token available, re-authenticating...")
            return self.authenticate()
        
        try:
            new_token = self.oauth.refresh_token(
                self.token_url,
                refresh_token=self.refresh_token,
                client_id=self.client_id,
                client_secret=self.client_secret
            )
            
            self.token = new_token
            self.access_token = new_token["access_token"]
            self.refresh_token = new_token.get("refresh_token", self.refresh_token)
            self.token_expiry = time.time() + new_token.get("expires_in", 3600)
            
            print("✓ Token refreshed successfully")
            return True
        except Exception as e:
            print(f"✗ Refresh failed: {e}")
            print("Re-authenticating...")
            return self.authenticate()
    
    def _ensure_authenticated(self):
        """Ensure we have a valid token, authenticate if needed"""
        if not self.access_token or self._is_token_expired():
            if self.refresh_token:
                self._refresh_access_token()
            else:
                self.authenticate()
    
    def get(self, url, **kwargs):
        """Make GET request with automatic authentication/refresh"""
        self._ensure_authenticated()
        headers = kwargs.pop('headers', {})
        headers['Authorization'] = f'Bearer {self.access_token}'
        headers['Content-Type'] = headers.get('Content-Type', 'application/json')
        return requests.get(url, headers=headers, **kwargs)
    
    def post(self, url, **kwargs):
        """Make POST request with automatic authentication/refresh"""
        self._ensure_authenticated()
        headers = kwargs.pop('headers', {})
        headers['Authorization'] = f'Bearer {self.access_token}'
        headers['Content-Type'] = headers.get('Content-Type', 'application/json')
        return requests.post(url, headers=headers, **kwargs)


In [3]:
# Create manager once (works for both new and existing sessions)
auth_manager = OAuthManager(
    client_id=os.getenv('AUTH_FLOW_CLIENT_ID'),
    client_secret=os.getenv('AUTH_FLOW_CLIENT_SECRET'),
    redirect_uri=os.getenv('REDIRECT_URI'),
    auth_url=os.getenv('AUTH_URL'),
    token_url=os.getenv("TOKEN_URL"),
    scope=os.getenv('SCOPE')
)

In [4]:
training_response = auth_manager.get("https://datacatalog-d.lanl.gov/server/ops_core_publication/ws_training_ops_core/views/i_ods_pa_cpnt_evthst")

Authenticating...
✗ Authentication failed


In [28]:
i_ods_pa_cpnt_evthst = json_normalize(training_response.json()['elements'])

In [ ]:
i_ods_pa_cpnt_evthst.head()

,rcd_type,utrain_uid,stud_id,cpnt_typ_id,cpnt_id,rev_dte,cmpl_stat_id,compl_dte,schd_id,cpnt_desc,...,esig_meaning_code_desc,cpnt_key,lst_upd_usr,lst_upd_tstmp,esig_comments,chklst_id,created_date,cpnt_evthst_key,cpnt_title,rcd_add_ts
0,CURRENT,4145725029,182151,COURSE,50695,1979-01-01T00:00:00,CR-COMPL,2009-07-08T00:00:00,NaN,RSAA 09-TA-55-13,...,None,7787.0,000005937,2011-06-17T16:37:46,None,None,NaN,NaN,RSAA 09-TA-55-13,2026-03-25T19:00:16
1,CURRENT,2323365496,181683,COURSE,41716,1979-01-01T00:00:00,CR-COMPL,2007-03-22T00:00:00,NaN,CLASSROOM:WCRRF RESPONSE MANUAL,...,None,12478.0,000009308,2011-06-17T16:37:06,None,None,NaN,NaN,WCRRF Emergency and Abnormal Response,2026-03-25T19:00:16
2,CURRENT,3349624369,181683,COURSE,41716,1979-01-01T00:00:00,CR-COMPL,2009-07-28T00:00:00,NaN,WCRRF RESPONSE MANUAL,...,None,12478.0,000009308,2011-06-17T16:37:06,None,None,NaN,NaN,WCRRF Emergency and Abnormal Response,2026-03-25T19:00:16
3,CURRENT,2576226358,181683,COURSE,41736,1979-01-01T00:00:00,CR-COMPL,2008-12-10T00:00:00,NaN,CAO:MANAGEMENT OBSERVATION & VERIFICATION,...,None,12409.0,000000000,2011-06-17T16:37:06,None,None,NaN,NaN,CAS: MANAGEMENT OBSERVATION & VERIFICATION,2026-03-25T19:00:16
4,CURRENT,2159409014,181683,COURSE,41748,1979-01-01T00:00:00,CR-COMPL,2009-11-30T00:00:00,NaN,RESPONDING TO A CORRECTIVE ACTION ASSIGNMENT,...,None,12348.0,000114477,2011-06-17T16:37:06,None,None,NaN,NaN,RESPONDING TO A CORRECTIVE ACTION ASSIGNMENT,2026-03-25T19:00:16


In [29]:
i_ods_pa_cpnt_evthst.to_csv('i_ods_pa_cpnt_evthst.csv')

In [10]:
# from IPython.display import HTML
# # Fetch HTML from Denodo webservice
# training_response.content
# display(HTML(training_response.text))


# Get Views

In [11]:
# First time in session - will trigger browser authentication
vdp_request = auth_manager.get("https://datacatalog-d.lanl.gov/denodo-data-catalog/public/api/views?serverId=1")
if vdp_request.status_code == 200:
    dataframe = json_normalize(vdp_request.json())
    vdpViews = json_normalize(vdp_request.json()).rename(columns={'name': 'Table-View Name','description':'vdp_description'})
    vdpViews = vdpViews[vdpViews['db'].str.startswith('dataportal')]
    dcStagingViews = vdpViews[vdpViews['path'].str.startswith('/02_staging_catalog/')]

# Read Export zip
export, pivot and rename columns based on the properties df

In [19]:
dataframe

,id,name,value,description,db,elementType,elementSubtype,path,lastModificationDate,fields,deleted
0,None,aa1,None,,dataportal,view,interface,/03_data_catalog/financial management & servic...,2025-07-07T14:46:00.000+00:00,None,False
1,None,aa1_locale,None,,dataportal,view,interface,/03_data_catalog/financial management & servic...,2025-07-07T14:46:02.000+00:00,None,False
2,None,aa2,None,,dataportal,view,interface,/03_data_catalog/financial management & servic...,2025-07-07T14:46:03.000+00:00,None,False
3,None,aa3,None,,dataportal,view,interface,/03_data_catalog/financial management & servic...,2025-07-07T14:46:05.000+00:00,None,False
4,None,aa4,None,,dataportal,view,interface,/03_data_catalog/financial management & servic...,2025-07-07T14:46:06.000+00:00,None,False
...,...,...,...,...,...,...,...,...,...,...,...
15149,None,zd_ws_costcode,None,,dataportal,view,interface,/03_data_catalog/waste management/waste compli...,2025-07-07T21:10:30.000+00:00,None,False
15150,None,zd_ws_epacode,None,,dataportal,view,interface,/03_data_catalog/waste management/waste compli...,2025-07-07T21:10:32.000+00:00,None,False
15151,None,zd_ws_uhc,None,,dataportal,view,interface,/03_data_catalog/waste management/waste compli...,2025-07-07T21:10:33.000+00:00,None,False
15152,None,zd_ws_workpath,None,,dataportal,view,interface,/03_data_catalog/waste management/waste compli...,2025-07-07T21:10:35.000+00:00,None,False


In [ ]:

url = "https://datacatalog-d.lanl.gov/denodo-data-catalog/public/api/configuration/metadata/export?assistedQuery=false&contentSearch=false&email=false&kerberos=false&permission=false&personalization=false&recommendations=false&requests=false&serverId=1"
headers = {
  'Accept': 'application/json',
   'Authorization': f'Bearer {token}'
}

devFile = auth_manager.post("POST", url, headers=headers, data=payload)
devFile.raise_for_status()

# Save the exported file
with open("denodo_export.zip", "wb") as f:
    f.write(devFile.content)


HTTPError: 500 Server Error:  for url: https://datacatalog-d.lanl.gov/denodo-data-catalog/public/api/configuration/metadata/export?assistedQuery=false&contentSearch=false&email=false&kerberos=false&permission=false&personalization=false&recommendations=false&requests=false&serverId=1

In [18]:
url = "https://datacatalog-d.lanl.gov/denodo-data-catalog/public/api/configuration/metadata/export?assistedQuery=false&contentSearch=false&email=false&kerberos=false&permission=false&personalization=false&recommendations=false&requests=false&serverId=1"
auth_manager.post(url)

<Response [500]>

In [ ]:


destination_directory = "denodo export"  # Optional: specify a directory for extraction

try:
    # Send a POST request to the URL to download the zip file content
    response = requests.request("POST", url, headers=headers, data=payload,stream=True)
    response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)

    # Create a BytesIO object to handle the zip file content in memory
    zip_in_memory = io.BytesIO(response.content)

    # Open the zip file from the in-memory object
    with zipfile.ZipFile(zip_in_memory, 'r') as zip_ref:
        # Extract all contents to the specified directory (or current directory if not specified)
        zip_ref.extractall(destination_directory)
        print(f"ZIP file downloaded and extracted to '{destination_directory}' successfully.")

except requests.exceptions.RequestException as e:
    print(f"Error downloading the ZIP file: {e}")
except zipfile.BadZipFile:
    print("Error: The downloaded file is not a valid ZIP file.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    print(f"An unexpected error occurred: {e}")

ZIP file downloaded and extracted to 'denodo export' successfully.
